# OpenDR 1.0 Case Study B: Transboundary Hydrology & Health
## Location: Baro-Akobo-Sobat Basin (Ethiopia / South Sudan border)

This notebook demonstrates the **OpenDR 1.0** multidimensional pipeline for flood resilience. We combine:
1. **Subsurface Intelligence:** GRACE Total Water Storage (TWS) anomalies to predict soil saturation.
2. **Surface Inundation:** Adaptive Otsu Thresholding of Sentinel-1 SAR to map flood extents.
3. **Humanitarian Exposure:** Integration with Building Footprints and Malaria Early Warning.

### Scientific References:
- **Hydrology:** Purdy & Famiglietti (2024), Markert et al. (2024)
- **Health:** Nekorchuk (2024) [EPIDEMIA Logic]

In [ ]:
import xarray as xr
import geopandas as gpd
import rioxarray
import matplotlib.pyplot as plt
import numpy as np
from shapely.geometry import shape
import sys
import os

# Ensure project root is in path for microservice access
sys.path.append(os.path.abspath('../'))
from src.microservices.flood_otsu import compute_flood_mask
from src.microservices.hydrology_grace import calculate_saturation_threshold
from src.microservices.health_epidemia import malaria_risk_forecast

print("OpenDR 1.0 Environmental Intelligence Modules Loaded.")

## 1. Subsurface Pre-cursor Analysis (GRACE)
Before surface water is visible, we check if the 'subsurface sponge' is saturated. A high TWS anomaly indicates that any subsequent rainfall will lead to immediate flash flooding.

In [ ]:
# Load Basin Boundary
basin = gpd.read_file('../data/sample/bas_basin_boundary.geojson')
basin_geom = basin.geometry.iloc[0]

# Analyze GRACE Gravimetry Anomalies
# Using NetCDF data from data/external/grace/ (Sample created during setup)
grace_data_path = '../data/external/grace/Baro_Akobo_tws.nc'

if os.path.exists(grace_data_path):
    status = calculate_saturation_threshold(grace_data_path, basin_geom)
    print(f"Basin Saturation Anomaly: {status['tws_anomaly_cm']:.2f} cm")
    print(f"Pre-cursor Flood Alert Active: {status['flood_precursor_alert']}")
else:
    print("Sample GRACE data not found. Run 'make data' first.")

## 2. Surface Water Extraction (Adaptive Otsu)
Using the Cloud-Native SAR sample, we apply the re-implemented Otsu thresholding logic to delineate active flood extent in the Machar Marshes.

In [ ]:
# Load Cloud-Optimized GeoTIFF (COG) of Sentinel-1 SAR
sar_path = '../data/sample/flood_mask_cog.tif'

# Execute the Adaptive Otsu microservice
flood_mask = compute_flood_mask(sar_path)

# Visualization
fig, ax = plt.subplots(1, 2, figsize=(15, 7))

raw_sar = xr.open_rasterio(sar_path)
raw_sar.plot(ax=ax[0], cmap='Greys_r', title="Sentinel-1 SAR (Backscatter)")
flood_mask.plot(ax=ax[1], cmap='Blues', title="Extracted Flood Extent (Otsu)")

plt.tight_layout()
plt.show()

## 3. Public Health Context (Malaria Risk)
Following the **EPIDEMIA** methodology, we correlate current inundation with Land Surface Temperature (LST) to predict mosquito breeding suitability.

In [ ]:
# Inputs: LST Anomaly from Tier 1 (Daymet) and Precipitation (CHIRPS)
current_lst = 28.5 # Degrees Celsius
monthly_precip = 120.0 # mm

health_alert = malaria_risk_forecast(current_lst, monthly_precip)

print(f"Malaria Breeding Suitability Index: {health_alert['malaria_suitability_index']}/2")
print(f"Integrated Alert Status: {health_alert['alert_level']}")

## 4. Final Risk Map & OGC API Preparation
We convert the flood mask to a vector and prepare it for Tier 4 (Mediation) storage in PostGIS.

In [ ]:
print("[*] Finalizing Tier 3 Intelligence Layer...")
print("[*] Generating GeoJSON for OGC API Feature delivery...")

# In a full pipeline, this would call src.processing.spatial_ops.push_to_postgis()
total_area_ha = (flood_mask.sum() * (100 * 100)) / 10000
print(f"Total Basin Inundation detected: {total_area_ha.values:.2f} Hectares.")
print("--- OpenDR 1.0: Case Study B Complete ---")